![Cloud-First](https://github.com/tulip-lab/SIT742/blob/develop/Jupyter/image/CloudFirst.png?raw=1)

# SIT742: Modern Data Science
**(Module 03: Big Data)**

**Session 3X: Data Wrangling I**

---

- Materials in this module support practical learning in modern data science, big data processing, data acquisition, and applied analytics.
- Materials may include adapted or referenced open-source resources. Keep attribution and licence notes where applicable.
- The public notebook collection is available in [SIT742](https://github.com/tulip-lab/sit742).
- If you find any issue or bug in this document, please submit an issue at [SIT742](https://github.com/tulip-lab/sit742/issues).
- Audience: Honours and Master's students using the Deakin SIT742 practical and self-learning materials.

Prepared by the SIT742 Teaching Team.

Maintained through the [TULIP Lab](https://www.tulip.academy) FLIP workflow.

---

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr>
<td align="left">Module context</td>
<td>This notebook is one component of the practical and self-learning materials for Module 03. The full module is normally completed across two two-hour practical sessions, together with the other notebooks listed in the SIT742 repository.</td>
</tr>
<tr>
<td align="left">Environment</td>
<td>Google Colab or local Jupyter with Python, pandas, NumPy, and the public AirCrashes data file</td>
</tr>
<tr>
<td align="left">Main output</td>
<td>A tidy AirCrashes DataFrame and a controlled runtime CSV output</td>
</tr>
<tr>
<td align="left">Related assessment</td>
<td>General practical skill development</td>
</tr>
</tbody>
</table>

</div>


<a id="1-overview-and-learning-goals"></a>
### 1. Overview and Learning Goals

Raw data is often delivered in a shape that looks like a table but does not behave like one. This session uses the `AirCrashes.csv` file to practise inspection, pattern recognition, deterministic parsing, type conversion, and quality checks.

By the end of this lab, students should be able to:

1. inspect why a raw file cannot be read as a clean CSV table;
2. recognise repeated record blocks in a line-based text file;
3. convert key-value blocks into a structured pandas DataFrame;
4. repair selected date and numeric fields with explicit checks; and
5. save generated outputs into a controlled runtime folder.


<a id="2-setup-and-data-files"></a>
### 2. Setup and Data Files

This notebook uses one public SIT742 file: `AirCrashes.csv`.

#### Option A: Google Colab / online execution

Keep `EXECUTION_MODE = "online"`. The setup cell downloads the public data file from GitHub into this notebook runtime.

#### Option B: Local repository execution

Use `EXECUTION_MODE = "local"` only when you have cloned the public SIT742 repository and are running the notebook from a location where the `Jupyter/data/` folder can be found.


In [1]:
from pathlib import Path
from urllib.request import urlretrieve   #DOWNLOADS A file from the internet
import re
import sys
import tempfile

import numpy as np
import pandas as pd

PUBLIC_DATA_BASE_URL = "https://raw.githubusercontent.com/tulip-lab/sit742/develop/Jupyter/data"
EXECUTION_MODE = "online"  # Use "online" for Google Colab; use "local" for a cloned SIT742 repository.

required_files = ["AirCrashes.csv"]
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="sit742_m03x_output_"))


def download_public_data(required_files):
    data_dir = Path(tempfile.mkdtemp(prefix="sit742_m03x_data_"))
    downloaded_paths = {}
    for filename in required_files:
        local_file = data_dir / filename
        urlretrieve(f"{PUBLIC_DATA_BASE_URL}/{filename}", local_file)
        downloaded_paths[filename] = local_file
    return data_dir, downloaded_paths


def find_local_data_dir(required_files):
    candidates = [
        Path.cwd() / "Jupyter" / "data",
        Path.cwd() / "data",
        Path.cwd().parent / "data",
        Path.cwd().parent.parent / "Jupyter" / "data",
    ]
    for candidate in candidates:
        if all((candidate / filename).exists() for filename in required_files):
            return candidate
    searched = "\n".join(str(candidate) for candidate in candidates)
    raise FileNotFoundError("Could not find the SIT742 public data folder. Searched:\n" + searched)


if EXECUTION_MODE == "online":
    DATA_DIR, data_paths_by_file = download_public_data(required_files)
elif EXECUTION_MODE == "local":
    DATA_DIR = find_local_data_dir(required_files)
    data_paths_by_file = {filename: DATA_DIR / filename for filename in required_files}
else:
    raise ValueError('EXECUTION_MODE must be "online" or "local".')

aircrashes_path = data_paths_by_file["AirCrashes.csv"]

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Data file:", aircrashes_path)
print("Output folder:", OUTPUT_DIR)


Python: 3.12.13
pandas: 2.2.2
Data file: /tmp/sit742_m03x_data_3_3ipu90/AirCrashes.csv
Output folder: /tmp/sit742_m03x_output_7frg0fb_


In [2]:
print("File size:", aircrashes_path.stat().st_size, "bytes")
print("First 680 characters:")
print(aircrashes_path.read_text(encoding="utf-8")[:680])


File size: 19975 bytes
First 680 characters:

Incident American Airlines Flight 11 involving a Boeing 767-223ER in 2001
Casualties,Extremely High
Total Dead,1692
Crew,11
Passengers,81
Ground,1600
Notes,No survivors
Type,INH
Reason,Attack
Location,New York - New York - US
Country,US
Phase,ENR
Date,2001-09-11
Latitude,40.7143528
Longitude,-74.0059731
Circumstances,Good Visibility by Day

Incident United Airlines Flight 175 involving a Boeing 767-222 in 2001
Casualties,Extremely High
Total Dead,965
Crew,9
Passengers,56
Ground,900
Notes,No survivors
Type,INH
Reason,Attack
Location,New York - New York - US
Country,USA
Phase,ENR
Date,2001-09-11
Latitude,40.7143528
Longitude,-74.0059731
Circumstances,Good Visibility by Day


<a id="3-inspect-the-raw-file"></a>
### 3. Inspect the Raw File

Try reading the file directly with `pandas.read_csv`. The result is not a useful table, because the first line does not contain a standard comma-separated header and the records are arranged as repeated key-value blocks.


In [4]:
direct_read = pd.read_csv(aircrashes_path)
print("Direct read shape:", direct_read.shape)
direct_read.head(20)


Direct read shape: (927, 1)


,Incident American Airlines Flight 11 involving a Boeing 767-223ER in 2001
Casualties,Extremely High
Total Dead,1692
Crew,11
Passengers,81
Ground,1600
Notes,No survivors
Type,INH
Reason,Attack
Location,New York - New York - US
Country,US


The visible pattern is:

- each record starts with an `Incident ...` line;
- the following lines use `Field,Value` pairs;
- blank lines separate incident records;
- each complete record should become one row in a clean DataFrame.


<a id="4-parse-key-value-blocks"></a>
### 4. Parse Key-Value Blocks

The parser below reads line by line. A new `Incident` line starts a new record, and ordinary key-value lines fill fields in the current record.


In [5]:
def parse_aircrash_blocks(path):
    records = []
    current_record = {}

    for raw_line in Path(path).read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line:
            continue

        if line.startswith("Incident "):
            if current_record:
                records.append(current_record)
            current_record = {"Incident": line.removeprefix("Incident ").strip()}
            continue

        if "," in line:
            key, value = line.split(",", 1)
            current_record[key.strip()] = value.strip()

    if current_record:
        records.append(current_record)

    return pd.DataFrame(records)


expected_columns = [
    "Incident",
    "Casualties",
    "Total Dead",
    "Crew",
    "Passengers",
    "Ground",
    "Notes",
    "Type",
    "Reason",
    "Location",
    "Country",
    "Phase",
    "Date",
    "Latitude",
    "Longitude",
    "Circumstances",
]

aircrashes_raw = parse_aircrash_blocks(aircrashes_path)
aircrashes_raw = aircrashes_raw.reindex(columns=expected_columns)

print("Parsed rows:", aircrashes_raw.shape[0])
print("Parsed columns:", aircrashes_raw.shape[1])
aircrashes_raw.head()


Parsed rows: 58
Parsed columns: 16


,Incident,Casualties,Total Dead,Crew,Passengers,Ground,Notes,Type,Reason,Location,Country,Phase,Date,Latitude,Longitude,Circumstances
0,American Airlines Flight 11 involving a Boeing...,Extremely High,1692,11,81,1600,No survivors,INH,Attack,New York - New York - US,US,ENR,2001-09-11,40.7143528,-74.0059731,Good Visibility by Day
1,United Airlines Flight 175 involving a Boeing ...,Extremely High,965,9,56,900,No survivors,INH,Attack,New York - New York - US,USA,ENR,2001-09-11,40.7143528,-74.0059731,Good Visibility by Day
2,Pan Am Flight 1736 and KLM Flight 4805 involvi...,Extremely High,583,23,560,0,Some survivors,COM,Accident,Tenerife - Spain,Spain,TOF,1977-03-27,28.2915637,-16.6291304,Bad Visibility by Day
3,Japan Airlines Flight 123 involving a Beoing 7...,Extremely High,520,15,505,0,Some survivors,COM,Accident,Ueno - Japan,Japan,ENR,1985-08-12,35.7089461,139.7742683,Bad Visibility by Night
4,Saudi Arabian Flight 763 and involving a Boein...,Extremely High,349,33,316,0,No survivors,,Accident,Charkhi Dadri - India,India,ENR,1996-11-12,28.6,76.2667,Bad Visibility by Night


In [ ]:
records = []
current_record = {}

for raw_line in Path(aircrashes_path).read_text(encoding="utf-8").splitlines():
    line = raw_line.strip()
    if not line:
        continue

    if line.startswith("Incident "):
        if current_record:
            records.append(current_record)
        current_record = {"Incident": line.removeprefix("Incident ").strip()}
        continue

    if "," in line:
        key, value = line.split(",", 1)
        current_record[key.strip()] = value.strip()

    print(current_record)


if current_record:
    records.append(current_record)



<a id="5-clean-types-and-dates"></a>
### 5. Clean Types and Dates

After parsing, convert numeric columns and parse dates. The raw file includes a few date strings that need explicit correction before they can be used reliably.


In [10]:
aircrashes = aircrashes_raw.copy()

numeric_columns = ["Total Dead", "Crew", "Passengers", "Ground", "Latitude", "Longitude"]
for column in numeric_columns:
    aircrashes[column] = pd.to_numeric(aircrashes[column], errors="coerce")

date_corrections = {
    "1989-19-09": "1989-09-19",
    "1890-07-08": "1980-07-08",
    "1984-06-06": "1994-06-06",
}
aircrashes["Date original"] = aircrashes["Date"]
aircrashes["Date"] = aircrashes["Date"].replace(date_corrections)
aircrashes["Date"] = pd.to_datetime(aircrashes["Date"], errors="coerce")

aircrashes["Incident year"] = pd.to_numeric(
    aircrashes["Incident"].str.extract(r"in (\d{4})\s*$")[0],
    errors="coerce",
)

year_mismatch = aircrashes[
    aircrashes["Incident year"].notna()
    & aircrashes["Date"].notna()
    & (aircrashes["Incident year"] != aircrashes["Date"].dt.year)
]

print("Missing values by selected column:")
print(aircrashes[["Total Dead", "Crew", "Passengers", "Ground", "Date", "Latitude", "Longitude"]].isna().sum())
print("Year mismatches after corrections:", len(year_mismatch))

tidy_path = OUTPUT_DIR / "AirCrashes_tidy.csv"
aircrashes.to_csv(tidy_path, index=False)

print("Tidy output:", tidy_path)
aircrashes.head()


Missing values by selected column:
Total Dead    0
Crew          1
Passengers    0
Ground        0
Date          0
Latitude      0
Longitude     0
dtype: int64
Year mismatches after corrections: 0
Tidy output: /tmp/sit742_m03x_output_7frg0fb_/AirCrashes_tidy.csv


,Incident,Casualties,Total Dead,Crew,Passengers,Ground,Notes,Type,Reason,Location,Country,Phase,Date,Latitude,Longitude,Circumstances,Date original,Incident year
0,American Airlines Flight 11 involving a Boeing...,Extremely High,1692,11.0,81,1600,No survivors,INH,Attack,New York - New York - US,US,ENR,2001-09-11,40.714353,-74.005973,Good Visibility by Day,2001-09-11,2001
1,United Airlines Flight 175 involving a Boeing ...,Extremely High,965,9.0,56,900,No survivors,INH,Attack,New York - New York - US,USA,ENR,2001-09-11,40.714353,-74.005973,Good Visibility by Day,2001-09-11,2001
2,Pan Am Flight 1736 and KLM Flight 4805 involvi...,Extremely High,583,23.0,560,0,Some survivors,COM,Accident,Tenerife - Spain,Spain,TOF,1977-03-27,28.291564,-16.629130,Bad Visibility by Day,1977-03-27,1977
3,Japan Airlines Flight 123 involving a Beoing 7...,Extremely High,520,15.0,505,0,Some survivors,COM,Accident,Ueno - Japan,Japan,ENR,1985-08-12,35.708946,139.774268,Bad Visibility by Night,1985-08-12,1985
4,Saudi Arabian Flight 763 and involving a Boein...,Extremely High,349,33.0,316,0,No survivors,,Accident,Charkhi Dadri - India,India,ENR,1996-11-12,28.600000,76.266700,Bad Visibility by Night,1996-11-12,1996


In [ ]:
#aircrashes[aircrashes["Crew"].isna()]


#aircrashes = aircrashes.dropna(subset=["Crew"])

#aircrashes["Crew"].fillna(
#     aircrashes["Crew"].median(),
#     inplace=True
# )

<a id="6-summarise-the-tidy-data"></a>
### 6. Summarise the Tidy Data

Use small summaries to check whether the resulting table behaves like a normal analysis dataset.


In [ ]:
summary = {
    "rows": len(aircrashes),
    "columns": len(aircrashes.columns),
    "total_recorded_deaths": int(aircrashes["Total Dead"].sum()),
    "earliest_date": aircrashes["Date"].min().date().isoformat(),
    "latest_date": aircrashes["Date"].max().date().isoformat(),
}

print(summary)
aircrashes.groupby("Country")["Total Dead"].sum().sort_values(ascending=False).head(10)


<a id="7-student-tasks"></a>
### 7. Student Tasks

Complete the following checks and short extensions.

1. Find the three incidents with the highest `Total Dead` values.
2. Count incidents by `Reason` and `Phase`.
3. Add a boolean column called `Has ground deaths`.
4. Save a filtered CSV containing only incidents after 1990.
5. Write one sentence explaining why direct CSV loading was not enough for this file.


In [ ]:
# Task workspace.
top_incidents = aircrashes.sort_values("Total Dead", ascending=False).head(3)
reason_phase_counts = aircrashes.groupby(["Reason", "Phase"]).size().reset_index(name="count")
aircrashes["Has ground deaths"] = aircrashes["Ground"] > 0
after_1990_path = OUTPUT_DIR / "AirCrashes_after_1990.csv"
aircrashes.loc[aircrashes["Date"].dt.year > 1990].to_csv(after_1990_path, index=False)

print(top_incidents[["Incident", "Total Dead", "Date"]])
print(reason_phase_counts.head())
print("Filtered output:", after_1990_path)


<a id="8-checks"></a>
### 8. Checks

Run these checks before moving on. If one fails, inspect the relevant transformation step.


In [ ]:
assert aircrashes.shape[0] == 58
assert set(expected_columns).issubset(set(aircrashes.columns))
assert aircrashes["Date"].notna().all()
assert year_mismatch.empty
assert aircrashes["Total Dead"].notna().all()
assert aircrashes[numeric_columns].isna().sum().max() <= 1
assert tidy_path.exists()
assert after_1990_path.exists()

print("M03X checks passed.")


<a id="9-reflection-and-references"></a>
### 9. Reflection and References

Reflection prompts:

1. Which assumption made direct `read_csv` fail for this file?
2. What evidence showed that the data was arranged in repeated blocks?
3. Which quality checks would you keep if this parser became part of a larger workflow?

References:

- pandas documentation: `read_csv`, `DataFrame`, `to_datetime`, `groupby`
- Python documentation: `pathlib.Path`
